In [26]:
import os
import re
from datetime import datetime
from itertools import chain
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import spacy
from spacy.tokens.doc import Doc
from spacy.tokens.span import Span
from spacy.tokens.token import Token
import coreferee
from coreferee.data_model import ChainHolder, Chain
from helpers import get_request
from bs4 import BeautifulSoup
from bs4.element import Tag

In [2]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("coreferee")

c:\Users\Adam\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_core_web_lg' (3.5.0) was trained with spaCy v3.5.0 and may not be 100% compatible with the current version (3.8.8). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [3]:
def get_text_from_file(path: str):
    text = ""
    
    try:
        with open(path, "r", encoding="utf8") as file:
            text = file.read()
    except Exception as e:
        print("Error reading file", path)
        print(e)
    finally:
        return text

In [4]:
articles_folder = "./articles"

In [5]:
article_files = os.listdir(articles_folder)
article_files = [file for file in article_files if file.endswith(".txt")]

In [6]:
texts = [get_text_from_file(f"{articles_folder}/{file}") for file in article_files]

df = pd.DataFrame(data={
    "file": article_files,
    "text": texts
})
df

,file,text
0,news_10-get-ready-for-a-november-to-remember-i...,"November is stacked. Normally, I would look to..."
1,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ..."
2,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...
3,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu..."
4,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...
...,...,...
876,news_zhang-mingyang-embracing-spotlight-ufc-sh...,Zhang Mingyang has all the makings of a potent...
877,news_zhang-mingyang-journey-main-event-light-h...,Zhang Mingyang first entered the UFC ecosystem...
878,news_zhang-weili-big-challenge-apple-vechain-u...,Zhang Weili knows all bout fighting in UFC tit...
879,news_zhang-weili-builds-her-case-goat-status.txt,


In [7]:
def already_split_gender_paragraphs(article_file: str):
    gender_paragraphs_folder = f"{articles_folder}/{article_file.replace('.txt', '')}"
    return os.path.exists(gender_paragraphs_folder)

In [ ]:
# Since this part of the pipeline is very resource intensive, 
# we only want to run it on articles that haven't already been split on gender.
# However, you can set this variable to True if you still want to run on everything.
run_on_all_files = False

if run_on_all_files:
    df_to_separate = df
else:
    df_to_separate = df[~df["file"].apply(already_split_gender_paragraphs)].copy()

df_to_separate

,file,text
0,news_10-get-ready-for-a-november-to-remember-i...,"November is stacked. Normally, I would look to..."
1,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ..."
2,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...
3,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu..."
4,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...
...,...,...
876,news_zhang-mingyang-embracing-spotlight-ufc-sh...,Zhang Mingyang has all the makings of a potent...
877,news_zhang-mingyang-journey-main-event-light-h...,Zhang Mingyang first entered the UFC ecosystem...
878,news_zhang-weili-big-challenge-apple-vechain-u...,Zhang Weili knows all bout fighting in UFC tit...
879,news_zhang-weili-builds-her-case-goat-status.txt,


In [9]:
# This takes a long time, so have patience
df_to_separate["doc"] = list(nlp.pipe(df_to_separate["text"]))

In [10]:
def get_people(doc: Doc):
    return [ent for ent in doc.ents if ent.label_ == "PERSON"]

In [11]:
df_to_separate["people"] = df_to_separate["doc"].apply(get_people)

In [12]:
all_people: list[Span] = list(chain.from_iterable(df_to_separate["people"]))
all_names = set(p.text for p in all_people)
full_names = set(name for name in all_names 
                 if 
                 len(name.split()) >= 2 
                 and len(name.split()) <= 3
                 and re.search("^[a-zA-Z\u00C0-\u017F\s’]+$", name) is not None)

full_names

{'Matt Frevola',
 'Jose Aldo',
 'Maurice Greene',
 'Curtis Millender',
 'Antonio Rogerio Nogueira',
 'Arman Tsarukyan',
 'Nathan Diaz',
 'Sam Greco',
 'Luis Pena',
 'Oban Elliott',
 'Chihiro Suzuki',
 'Edson Barboza',
 'Will Santiago',
 'Antonio Carlos Junior',
 'Bubba McDaniel',
 'Deiveson Figueiredo',
 'Ryan Hall',
 'Brazilian Ketlen Vieira',
 'Valter Walker’s',
 'Paul Craig',
 'Court McGee',
 'Alden Coria',
 'Irene Aldana',
 'Chris Duncan',
 'Kaik Brito',
 'Pat Healy',
 'Ken MacKenzie',
 'Hamdy Abdelwahab',
 'Montel Jackson',
 'Alex Caceres',
 'Jon Jones',
 'Claressa Shields',
 'Kevin\xa0Schuster',
 'Weston Wilson',
 'Adam Borics',
 'Shara Magomedov',
 'Logan Storley',
 'Charles Olivera',
 'Beating Allen',
 'Bartosz Szewczyk',
 'Marvin Vettori',
 'John Philips',
 'Soren Bak',
 'Askar Askarov',
 'Robbie Lawler',
 'Aljamain Sterling',
 'Kid Yamamoto',
 'Cory McKenna',
 'Andrew Thompson',
 'Jessica Andrade',
 'Randy Brown\n',
 'Aspen Ladd',
 'Victoria Leonardo',
 'Damon Wilson',
 'Albe

In [13]:
def is_woman_tag(tag: Tag):
    text = tag.get_text().strip().lower()
    return "woman" in text or "women" in text

def find_gender_on_ufc(name: str):
    name_query = "-".join(name.split()).replace('’', ' ')
    url = f"https://www.ufc.com/athlete/{name_query}"

    try:
        response = get_request(url)
        soup = BeautifulSoup(response.text, "html.parser")

        tags: list[Tag] = soup.find_all("p", "hero-profile__tag")
        if not tags:
            return None

        is_woman = any([is_woman_tag(tag) for tag in tags])
        if is_woman:
            return "woman"
        else:
            return "man"
    except Exception as e:
        print("Error getting gender of", name)
        print(e)

In [ ]:
# Set this to False if you want to fetch the genders from the UFC website
use_saved_gender_map = True
genders_map_file = "./genders_map_13-01-26_19-50-58.csv"

if use_saved_gender_map and os.path.exists(genders_map_file):
    name_gender_pairs_df = pd.read_csv(genders_map_file)
    genders_map: dict[str, str] = dict(zip(name_gender_pairs_df["name"], name_gender_pairs_df["gender"]))
else:
    def get_name_gender_pair_on_ufc(name: str):
        return (name, find_gender_on_ufc(name))

    with ThreadPoolExecutor() as executor:
        name_gender_pairs = [
            (n, g)
            for n, g in executor.map(get_name_gender_pair_on_ufc, full_names)
            if g is not None
        ]

    genders_map = dict(name_gender_pairs)

    pd.DataFrame(data={
        "name": genders_map.keys(),
        "gender": genders_map.values(),
    })\
        .set_index("name")\
        .to_csv(f"./genders_map_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv")

genders_map

{'Matt Frevola': 'man',
 'Jose Aldo': 'man',
 'Maurice Greene': 'man',
 'Curtis Millender': 'man',
 'Antonio Rogerio Nogueira': 'man',
 'Arman Tsarukyan': 'man',
 'Luis Pena': 'man',
 'Oban Elliott': 'man',
 'Edson Barboza': 'man',
 'Will Santiago': 'man',
 'Antonio Carlos Junior': 'man',
 'Bubba McDaniel': 'man',
 'Deiveson Figueiredo': 'man',
 'Ryan Hall': 'man',
 'Paul Craig': 'man',
 'Court McGee': 'man',
 'Alden Coria': 'man',
 'Irene Aldana': 'woman',
 'Chris Duncan': 'man',
 'Kaik Brito': 'man',
 'Pat Healy': 'man',
 'Hamdy Abdelwahab': 'man',
 'Montel Jackson': 'man',
 'Alex Caceres': 'man',
 'Jon Jones': 'man',
 'Shara Magomedov': 'man',
 'Marvin Vettori': 'man',
 'Askar Askarov': 'man',
 'Robbie Lawler': 'man',
 'Aljamain Sterling': 'man',
 'Cory McKenna': 'woman',
 'Jessica Andrade': 'woman',
 'Randy Brown\n': 'man',
 'Aspen Ladd': 'woman',
 'Victoria Leonardo': 'woman',
 'Alberto Montes': 'man',
 'Albert Tadevosyan': 'man',
 'Brad Tavares': 'man',
 'Shamil Gaziev': 'man',
 

In [15]:
df_test = df_to_separate[df_to_separate["file"] == "news_bonus-coverage-2025-performance-fight-of-the-night.txt"]
df_test

,file,text,doc,people
90,news_bonus-coverage-2025-performance-fight-of-...,What a way to close out 2025. In a year define...,"(What, a, way, to, close, out, 2025, ., In, a,...","[(Jamey, -, Lyn, Horth), (Tereza, Bledá), (Ste..."


In [27]:
def get_chain_gender_from_pronouns(chain: Chain, doc: Doc):
    for mention in chain:
        token = doc[mention.root_index]

        if token.lemma_ == "he" or token.lemma_ == "his":
            return "man"
        elif token.lemma_ == "she" or token.lemma_ == "her" or token.lemma_ == "hers":
            return "woman"

    return None

In [ ]:
def get_coreference_genders(row: pd.Series):
    doc: Doc = row["doc"]

    people: set[str] = set([person.text for person in row["people"]])

    doc_genders_map: dict[str, str] = {}

    for person in people:
        if person in genders_map.keys():
            doc_genders_map[person] = genders_map[person]

            name_components = person.split()
            
            if len(name_components) >= 2:
                for component in name_components:
                    doc_genders_map[component] = genders_map[person]

    chains: ChainHolder = doc._.coref_chains
    
    def loop():
        for chain in chains:
            chain_pronouns_gender = get_chain_gender_from_pronouns(chain, doc)

            # If the chain explicitly has gendered pronouns,
            # then we can mark the entire chain with that gender.
            # Otherwise, we need to look it up in the map
            if chain_pronouns_gender == "man":
                for mention in chain:
                    token = doc[mention.root_index]
                    yield (token, "man")
            elif chain_pronouns_gender == "woman":
                for mention in chain:
                    token = doc[mention.root_index]
                    yield (token, "woman")
            else:
                main_item_mention = chain[chain.most_specific_mention_index]
                main_item = doc[main_item_mention.root_index]
                
                if main_item.text in doc_genders_map:
                    for mention in chain:
                        token = doc[mention.root_index]
                        yield (token, doc_genders_map[main_item.text])
    
    return list(loop())

In [30]:
df_to_separate["coreference_genders"] = df_to_separate.apply(get_coreference_genders, axis="columns")

In [31]:
def get_people_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    people: list[Span] = row["people"]
    coreference_genders: list[tuple[Token, str]] = row["coreference_genders"]

    people_with_gender = [p for p in people if genders_map.get(p.text) == gender]
    coreferences_with_gender = [p for p, g in coreference_genders if g == gender]
    coreferences_with_gender = [Span(doc, p.i, p.i + 1) for p in coreferences_with_gender]
    
    return set(people_with_gender + coreferences_with_gender)

In [32]:
df_to_separate["men"] = df_to_separate.apply(get_people_with_gender, args=("man",), axis="columns")
df_to_separate["women"] = df_to_separate.apply(get_people_with_gender, args=("woman",), axis="columns")

In [33]:
def get_paragraph_boundaries(doc: Doc):
    line_breaks = [token for token in doc if token.text == "\n"]
    line_break_indexes = [lb.i for lb in line_breaks]
    paragraph_boundaries = list(zip([0] + line_break_indexes, line_break_indexes + [len(doc)]))
    return paragraph_boundaries

In [34]:
df_to_separate["paragraph_boundaries"] = df_to_separate["doc"].apply(get_paragraph_boundaries)

In [35]:
def count_genders_in_paragraphs(row: pd.Series, gender: str):
    people_with_gender: set[Span] = row[gender]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]

    def count_gender_in_paragraph(boundary: tuple[int, int]):
        start, end = boundary
        count = 0

        for person in people_with_gender:
            if person.end >= end:
                continue
            elif person.start < start:
                continue
            
            count += 1

        return count

    return [count_gender_in_paragraph(boundary) for boundary in paragraph_boundaries]

In [36]:
df_to_separate["men_per_paragraph"] = df_to_separate.apply(count_genders_in_paragraphs, args=("men",), axis="columns")
df_to_separate["women_per_paragraph"] = df_to_separate.apply(count_genders_in_paragraphs, args=("women",), axis="columns")

In [37]:
def assign_gender_to_paragraphs(row: pd.Series):
    men_per_paragraph: list[int] = row["men_per_paragraph"]
    women_per_paragraph: list[int] = row["women_per_paragraph"]

    def choose_gender(m: int, f: int):
        if m == 0 and f == 0:
            return None
        elif m > f:
            return "man"
        elif f > m:
            return "woman"
        else: 
            return "equal"

    return [choose_gender(m, f) for m, f in zip(men_per_paragraph, women_per_paragraph)]

In [38]:
df_to_separate["paragraph_genders"] = df_to_separate.apply(assign_gender_to_paragraphs, axis="columns")

In [39]:
def get_paragraphs_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]
    paragraph_genders: list[str | None] = row["paragraph_genders"]

    paragraphs_indexes_with_gender = [i for i, g in enumerate(paragraph_genders) if g == gender]
    paragraphs_boundaries_with_gender = [paragraph_boundaries[i] for i in paragraphs_indexes_with_gender]
    paragraphs_with_gender = [doc[start:end] for start, end in paragraphs_boundaries_with_gender]

    return paragraphs_with_gender

In [40]:
df_to_separate["man_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("man",), axis="columns")
df_to_separate["woman_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("woman",), axis="columns")
df_to_separate["equal_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("equal",), axis="columns")
df_to_separate["genderless_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=(None,), axis="columns")
df_to_separate

,file,text,doc,people,coreference_genders,men,women,paragraph_boundaries,men_per_paragraph,women_per_paragraph,paragraph_genders,man_paragraphs,woman_paragraphs,equal_paragraphs,genderless_paragraphs
0,news_10-get-ready-for-a-november-to-remember-i...,"November is stacked. Normally, I would look to...","(November, is, stacked, ., Normally, ,, I, wou...","[(Ketlen, Vieira), (Norma, Dumont), (Vieira), ...","[(Vieira, woman), (Vieira, woman), (her, woman...","{(Shavkat, Rakhmonov), (Garcia), (Morales), (h...","{(Erin, Blanchfield), (Pennington), (her), (Du...","[(0, 67), (67, 103), (103, 112), (112, 153), (...","[0, 0, 0, 0, 0, 0, 0, 5, 7, 9, 0, 3, 6, 7, 0, ...","[0, 0, 0, 4, 10, 8, 2, 0, 0, 0, 0, 0, 0, 0, 0,...","[None, None, None, woman, woman, woman, woman,...","[(\n, Steve, Garcia, and, David, Onama, clash,...","[(\n, You, know, it, ’s, a, loaded, month, whe...",[],"[(November, is, stacked, ., Normally, ,, I, wo..."
1,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ...","(Buckle, up, ,, kids, ,, because, the, June, s...","[(Mario, Bautista), (Bautista), (Ricky, Simon)...","[(Bautista, man), (Bautista, man), (him, man),...","{(his), (Nurmagomedov), (Brandon, Royval), (Mo...","{(Harrison), (she), (Namajunas), (Pena), (Ketl...","[(0, 14), (14, 94), (94, 164), (164, 174), (17...","[0, 0, 0, 0, 0, 2, 7, 3, 1, 0, 0, 0, 0, 0, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 7, 8, 1, 0, ...","[None, None, None, None, None, man, man, man, ...","[(\n, A, couple, fights, before, the, bantamwe...","[(\n, Julianna, Peña, seeks, to, successfully,...",[],"[(Buckle, up, ,, kids, ,, because, the, June, ..."
2,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...,"(There, are, all, kinds, of, different, ways, ...","[(Bo, Nickal), (Gerald, Meerschaert), (Kevin, ...","[(Holland, man), (his, man), (he, man), (conte...","{(his), (his), (Sandhagen), (Gilbert, Burns), ...","{(Jasudavicius), (Shevchenko), (She), (Manon, ...","[(0, 41), (41, 78), (78, 133), (133, 172), (17...","[0, 0, 0, 0, 0, 0, 1, 7, 7, 2, 4, 12, 7, 3, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, ...","[None, None, None, None, None, None, man, man,...","[(\n, While, both, Reinier, de, Ridder, and, B...","[(\n, Jessica, Andrade, and, Jasmine, Jasudavi...",[],"[(There, are, all, kinds, of, different, ways,..."
3,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu...","(On, Saturday, ,, February, 8, ,, the, UFC, re...","[(Alexander, Volkanovski), (Adesanya), (Dan, H...","[(Weili, woman), (her, woman), (Velasquez, man...","{(his), (Whittaker), (his), (Whittaker), (Whit...","{(Holm), (Reneau), (person), (Ronda, Rousey), ...","[(0, 30), (30, 67), (67, 159), (159, 220), (22...","[0, 0, 3, 2, 4, 5, 3, 5, 9, 5, 3, 0, 0, 0, 0, ...","[0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[None, None, man, woman, man, man, man, man, m...","[(\n, Over, the, years, ,, the, stops, in, Syd...","[(\n, As, we, ready, to, see, Zhang, Weili, de...",[],"[(On, Saturday, ,, February, 8, ,, the, UFC, r..."
4,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...,"(Fourteen, individuals, have, held, the, UFC, ...","[(Dave, Menne), (Gil, Castillo), (Dricus, Du, ...","[(Franklin, man), (Franklin, man), (ambassador...","{(He), (Chris, Leben), (he), (he), (Rich, Fran...",{},"[(0, 67), (67, 118), (118, 166), (166, 181), (...","[4, 0, 0, 3, 0, 7, 7, 6, 6, 3, 0, 4, 4, 10, 1,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[man, None, None, man, None, man, man, man, ma...","[(Fourteen, individuals, have, held, the, UFC,...",[],[],"[(\n, While, it, never, profiled, as, the, gla..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876,news_zhang-mingyang-embracing-spotlight-ufc-sh...,Zhang Mingyang has all the makings of a potent...,"(Zhang, Mingyang, has, all, the, makings, of, ...","[(Zhang, Mingyang), (Brendson, Ribeiro), (Ozzy...

In [41]:
def save_separate_paragraphs(row: pd.Series):
    original_file: str = row["file"]
    new_folder = original_file.replace(".txt", "")

    os.makedirs(f"./articles/{new_folder}", exist_ok=True)

    man_paragraphs: list[Span] = row["man_paragraphs"]
    man_text = "\n".join([p.text for p in man_paragraphs])

    with open(f"./articles/{new_folder}/man.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(man_text)

    woman_paragraphs: list[Span] = row["woman_paragraphs"]
    woman_text = "\n".join([p.text for p in woman_paragraphs])

    with open(f"./articles/{new_folder}/woman.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(woman_text)

    equal_paragraphs: list[Span] = row["equal_paragraphs"]
    equal_text = "\n".join([p.text for p in equal_paragraphs])

    with open(f"./articles/{new_folder}/equal.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(equal_text)

    genderless_paragraphs: list[Span] = row["genderless_paragraphs"]
    genderless_text = "\n".join([p.text for p in genderless_paragraphs])

    with open(f"./articles/{new_folder}/genderless.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(genderless_text)

In [42]:
df_to_separate.apply(save_separate_paragraphs, axis="columns")

0      None
1      None
2      None
3      None
4      None
       ... 
876    None
877    None
878    None
879    None
880    None
Length: 881, dtype: object